In [0]:
source_path = "/Volumes/workspace/default/metricmind_raw/dataco/"

display(dbutils.fs.ls(source_path))

In [0]:
df = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv(source_path)
)

display(df)

In [0]:
df.printSchema()

In [0]:
raw_table_path = "/Volumes/workspace/default/metricmind_raw/delta/dataco"
(
    df.write
    .format("delta")
    .option("delta.columnMapping.mode","name")
    .option("delta.minReaderVersion","2")
    .option("delta.minWriterVersion","5")
    .mode("overwrite")
    .save(raw_table_path)
)
display(dbutils.fs.ls(raw_table_path))

In [0]:
raw_df = spark.read.format("delta").load(raw_table_path)

display(raw_df)

In [0]:
raw_df.printSchema()

In [0]:
print("Raw Delta row count:",raw_df.count())

In [0]:
print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

In [0]:
from pyspark.sql.functions import col, sum, when

missing_df = raw_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
])

display(missing_df)

In [0]:
duplicate_count = raw_df.count() - raw_df.dropDuplicates().count()

print("Duplicate rows:", duplicate_count)